# E07_RacingCar — SAC, максимизация награды и энтропии

**Урок:** 3.1 «SAC — Soft Actor-Critic» и проект-3 «Гоночный агент»
курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)
**Что реализуется:** Soft Actor-Critic с двумя критиками и автоподстройкой α.
**Ожидаемый результат:** **≥ 80 %** заездов замыкают круг (награда ≈ 13).
**Время обучения:** около 18 минут на сид (`QUICK_RUN = True` — около двух минут).

## Что здесь принципиально нового

Это первый **off-policy** метод лаборатории с непрерывными действиями.
A2C, PPO и REINFORCE выбрасывают собранные данные после одного обновления:
они описывали политику, которой больше нет. SAC хранит переходы в буфере
и переиспользует каждый многократно.

Выбор метода здесь не вкусовой. Шаг среды в этой задаче дорог: заезд длится
до 400 решений, а авария обрывает его через десяток — данные добываются
медленно и неравномерно. Это и есть практическая ниша off-policy методов.

## Одна поправка, из которой следует всё остальное

SAC максимизирует не сумму наград, а сумму наград **плюс энтропию политики**:

$$J(\pi) = \sum_t \mathbb{E}\bigl[r(s_t, a_t) + \alpha\,H(\pi(\cdot|s_t))\bigr]$$

Агенту выгодно быть настолько случайным, насколько это не мешает набирать
награду. Разведка встроена в цель, а не приделана снаружи ε-жадностью,
поэтому в конфиге этого примера ε равен нулю и не используется вовсе.

## Требования

```powershell
python scripts\build_env.py E07_RacingCar
```

## 1. Окружение

In [1]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

python        3.10.11
torch         2.2.1+cu121  CUDA 12.1
mlagents_envs 1.2.0.dev0
устройство    cuda
сид           0


## 2. Конфигурация

`SACConfig` — свойства **метода**, `SACTrainConfig` — свойства **процесса**
(буфер, разогрев, частота обновлений). Значения совпадают с
`configs/E07_RacingCar__sac.yaml`.

Два параметра процесса, которых не было ни у одного предыдущего метода:

- `learning_starts` — сколько шагов собирать **равномерно случайными**
  действиями до первого обновления. Обучать критика на пустом буфере
  бессмысленно: он выучит статистику стартовой политики и будет мешать актору.
- `gradient_steps` — сколько обновлений делать на один шаг среды. Единица —
  обычный режим SAC: за шаг приходит 8 переходов (по числу арен), и один
  градиентный шаг на них — умеренный темп.

**Шаг обучения постоянный**, в отличие от on-policy методов. Проблемы T-13
(«шум критика в нормированных преимуществах разрушает политику») у SAC нет:
преимущества здесь не считаются вовсе, а энтропийная часть цели сама
удерживает политику от вырождения.

In [2]:
QUICK_RUN = True   # True — быстрая проверка (~2 мин); False — полное обучение (~18 мин)

from labrl.algos.sac import SACConfig
from labrl.train.sac import SACTrainConfig

TOTAL_STEPS = 6_000 if QUICK_RUN else 60_000

ALGO_CFG = SACConfig(
    gamma=0.99,
    learning_rate=3e-4,      # один шаг на актора, критиков и температуру
    batch_size=256,
    tau=0.005,               # мягкое обновление целевых критиков
    init_alpha=0.2,
    autotune_alpha=True,     # целевая энтропия по умолчанию -dim(A) = -2
    max_grad_norm=0.0,       # без клиппинга: у SAC он обычно не требуется
)

TRAIN_CFG = SACTrainConfig(
    total_steps=TOTAL_STEPS,
    buffer_size=300_000,     # весь опыт полного прогона помещается целиком
    # Разогрев НЕ сжимается пропорционально быстрому прогону лишь частично:
    # на 6000 шагов 5000 разогрева не оставили бы обучению ничего.
    learning_starts=1_000 if QUICK_RUN else 5_000,
    train_freq=1,
    gradient_steps=1,
    eval_every_steps=max(1, TOTAL_STEPS // 5),
    eval_episodes=20,
    # Круг и авария оба завершают эпизод, поэтому правило `terminated`
    # их не различает; награда различает однозначно: круг ≈ 13, авария = −1.
    success_threshold=10.0,
)

print(f"бюджет: {TOTAL_STEPS} шагов сбора, разогрев {TRAIN_CFG.learning_starts}, "
      f"обновлений: {max(0, TOTAL_STEPS - TRAIN_CFG.learning_starts) * TRAIN_CFG.gradient_steps}")

бюджет: 6000 шагов сбора, разогрев 1000, обновлений: 5000


## 3. Подключение среды

In [3]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E07_RacingCar"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")

behavior: E07_RacingCar?team=0
наблюдения (2 сенсоров):
  obs_0: shape=(27,)  имя сенсора: 'RayPerceptionSensor'
  obs_1: shape=(8,)  имя сенсора: 'VectorSensor_size8'
действия: непрерывных 2, дискретные ветки ()

арен (параллельных сред): 8


## 4. Инспекция пространств

Наблюдений два: 9 лучей по 3 числа (27) и вектор состояния машины (8).

In [4]:
OBS_DIM = vec.single_obs_dim          # склейка всех сенсоров: 27 + 8
ACTION_DIM = int(vec.action_spec.continuous_size)

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"obs_dim (склейка): {OBS_DIM}")
print(f"непрерывных:       {ACTION_DIM}")
print(f"\nobs_0 (лучи, 27):   {np.array2string(obs[0][0], precision=2)}")
print(f"obs_1 (вектор, 8):  {np.array2string(obs[1][0], precision=2)}")
print("  obs_1: [0] скорость, [1:3] боковая и продольная составляющие,")
print("         [3] угловая скорость, [4:6] направление на точку, [6] дистанция, [7] доля круга")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=8; discrete_branches=; continuous_size=2; max_step=2000 -->
assert [tuple(s) for s in vec.obs_shapes] == [(27,), (8,)], (
    f"порядок или размер сенсоров не тот: {vec.obs_shapes}")
assert OBS_DIM == 35, f"obs_dim {OBS_DIM} != 35"
assert ACTION_DIM == 2, f"непрерывных действий {ACTION_DIM} != 2 (ENV_SPEC.md)"
assert len(vec.action_spec.discrete_branches) == 0
print("\nразмерности и порядок сенсоров совпадают с ENV_SPEC.md")

obs_shapes:        [(27,), (8,)]
obs_dim (склейка): 35
непрерывных:       2

obs_0 (лучи, 27):   [1.   0.   0.13 1.   0.   0.11 1.   0.   0.19 1.   0.   0.1  1.   0.
 0.34 1.   0.   0.11 1.   0.   0.13 1.   0.   0.12 1.   0.   0.09]
obs_1 (вектор, 8):  [ 0.    0.    0.    0.   -0.53  0.85  0.12  0.  ]
  obs_1: [0] скорость, [1:3] боковая и продольная составляющие,
         [3] угловая скорость, [4:6] направление на точку, [6] дистанция, [7] доля круга

размерности и порядок сенсоров совпадают с ENV_SPEC.md


## 5. Нейросети

Сетей здесь **три** (плюс две целевые копии, которые алгоритм создаёт сам).

**Актор** — squashed-гауссова политика: выдаёт μ и σ, действие берётся как
$a = \tanh(\mu + \sigma\varepsilon)$. Тангенс нужен, чтобы действие попадало
в диапазон Unity без обрезки; за это приходится платить поправкой в $\log\pi$
(она внутри `SquashedGaussianPolicy.sample`).

**Два критика** $Q(s,a)$ — и они обязаны быть инициализированы **независимо**.
Минимум из двух одинаковых оценок не гасил бы переоценку, а повторял бы её.

Активация `relu`, а не `tanh`: у критиков нет требования гладкости, а обучаются
они быстрее. Сети шире (256), чем в `E04`/`E06`: каждая решает более простую
задачу, чем актор PPO, но их четыре.

In [5]:
from labrl.nets.squashed_gaussian import SquashedGaussianPolicy


class MyPolicy(SquashedGaussianPolicy):
    """Актор: obs (B, 35) -> tanh(μ) (B, 2), плюс sample() с log π."""

    def __init__(self, obs_dim: int, action_dim: int, hidden: int = 256):
        super().__init__(obs_dim, action_dim, (hidden, hidden), "relu")


class MyCritic(nn.Module):
    """Критик: (obs (B, 35), action (B, 2)) -> Q (B,)."""

    def __init__(self, obs_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim + action_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs, action):
        return self.body(torch.cat([obs, action], dim=1)).squeeze(-1)


policy_net = MyPolicy(OBS_DIM, ACTION_DIM)
q1_net = MyCritic(OBS_DIM, ACTION_DIM)
q2_net = MyCritic(OBS_DIM, ACTION_DIM)

with torch.no_grad():
    probe_obs = torch.zeros(3, OBS_DIM)
    probe_act = torch.zeros(3, ACTION_DIM)
    assert policy_net(probe_obs).shape == (3, ACTION_DIM), "актор обязан выдавать (B, action_dim)"
    assert q1_net(probe_obs, probe_act).shape == (3,), "критик обязан выдавать (B,)"
    # Независимая инициализация — не формальность: на этом стоит весь смысл
    # минимума из двух оценок.
    first = next(q1_net.parameters()).flatten()[:5]
    second = next(q2_net.parameters()).flatten()[:5]
    assert not torch.allclose(first, second), "критики инициализированы одинаково"

print(policy_net)
print(f"\nпараметров актора:   {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров критика:  {sum(p.numel() for p in q1_net.parameters()):,} (×2, плюс 2 целевые копии)")

MyPolicy(
  (body): Sequential(
    (0): Linear(in_features=35, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=4, bias=True)
  )
)

параметров актора:   76,036
параметров критика:  75,777 (×2, плюс 2 целевые копии)


## 6. Теория и математика метода

Цель критиков — soft Bellman backup: к обычной цели добавлена энтропия
следующего состояния, а из двух целевых оценок берётся **минимум**:

$$y = r + \gamma\,(1 - \mathrm{terminated})\Bigl[\min_{i=1,2}\bar Q_i(s', a')
- \alpha \log \pi(a'|s')\Bigr], \qquad a' \sim \pi(\cdot|s')$$

Цель актора — максимум Q при штрафе за детерминированность:

$$\mathcal{L}(\pi) = \mathbb{E}\bigl[\alpha \log \pi(a|s) - \min_{i} Q_i(s, a)\bigr]$$

Температура настраивается сама, на целевую энтропию $\bar H = -\dim(A)$:

$$\mathcal{L}(\alpha) = -\mathbb{E}\bigl[\log\alpha\,(\log \pi(a|s) + \bar H)\bigr]$$

**Почему минимум двух критиков.** Бутстрэп-обучение Q систематически завышает
ценность: в цели стоит максимум по зашумлённым оценкам, а максимум шума смещён
вверх. Минимум двух независимых оценок — намеренный пессимизм (приём из TD3).

**Почему $(1 - \mathrm{terminated})$, а не «эпизод кончился».** Будущее
обнуляется только при истинном завершении. Обрыв по `MaxStep` — не терминальное
состояние, там бутстрэппинг обязателен (требование 8.3). В этой среде разница
особенно наглядна: авария — настоящий терминал, а исчерпание времени — нет.

**Почему оптимизируется $\log\alpha$, а не $\alpha$.** Температура обязана быть
положительной; на её логарифм ограничений нет, и оптимизатор не может вывести
её в бессмысленную область.

Ниже — исходный код обновления, тот самый, что выполняется при обучении.

In [6]:
import inspect

from labrl.algos.sac import SAC

print(inspect.getsource(SAC.update))

    def update(self, batch: Batch) -> dict[str, float]:
        """Единственное место, где меняются параметры (требование 8.7)."""
        obs = torch.as_tensor(batch.obs, dtype=torch.float32, device=self.device)
        next_obs = torch.as_tensor(batch.next_obs, dtype=torch.float32, device=self.device)
        action = torch.as_tensor(batch.action, dtype=torch.float32, device=self.device)
        reward = torch.as_tensor(batch.reward, dtype=torch.float32, device=self.device)
        terminated = torch.as_tensor(batch.terminated, dtype=torch.bool, device=self.device)

        alpha = self.alpha

        # --- критики ------------------------------------------------------
        with torch.no_grad():
            next_action, next_log_prob = self.policy_net.sample(next_obs)
            # Минимум из двух целевых критиков гасит систематическую переоценку.
            next_q = torch.min(
                self.q1_target(next_obs, next_action),
                self.q2_target(next_obs, next_ac

## 7. Цикл обучения

Цикл — `labrl.train.sac.train_sac`. От on-policy цикла он отличается тремя
вещами: буфер воспроизведения, разогрев случайными действиями и обновление
по времени, а не по заполнению роллаута.

In [7]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.sac import train_sac

algo = SAC(policy_net, q1_net, q2_net, ALGO_CFG, device=DEVICE)
run = create_run_dir(ENV_ID, "sac", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_sac(
    vec=vec, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.3f}, кругов {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, переходов: {result.transitions}, "
      f"обновлений: {result.updates}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

каталог прогона: C:\unity-ml-agents-lab\results\E07_RacingCar\sac\20260816-095534_seed0



  шаг    1200: награда  13.014, кругов 100%


  шаг    2400: награда  13.025, кругов 100%


  шаг    3600: награда  13.026, кругов 100%


  шаг    4800: награда  13.032, кругов 100%


  шаг    6000: награда  13.034, кругов 100%

эпизодов: 921, переходов: 42867, обновлений: 5804
время: 129.4 с
обязательные теги схемы 11.2 без данных: нет


## 8. Оценка и кривая обучения

Третий график — температура α. По ней видно работу автоподстройки: пока
политика слишком детерминирована, α растёт и возвращает разведку; когда
разведки достаточно, α падает и вес награды в цели растёт.

Кривая награды в этой среде **ступенчатая**: пока агент не научился проходить
поворот, почти все заезды кончаются аварией (−1), и средняя награда стоит
на месте. Как только круг начинает замыкаться, оценка скачком уходит к 13.

In [8]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.episode_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].plot(smoothed)
axes[0].set_title(f"Награда за эпизод (скользящее среднее по {window})")
axes[0].set_xlabel("эпизод"); axes[0].set_ylabel("награда"); axes[0].grid(alpha=0.3)

steps = [s for s, _, _ in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]
success = [s for _, _, s in result.eval_history]
axes[1].plot(steps, rewards, marker="o", label="награда")
axes[1].axhline(13.0, color="g", ls=":", lw=1, label="полный круг ≈ 13")
axes[1].axhline(-1.0, color="r", ls=":", lw=1, label="авария = −1")
axes[1].set_title("Eval/Mean Reward"); axes[1].set_xlabel("шаг сбора")
axes[1].grid(alpha=0.3); axes[1].legend()

axes[2].plot(steps, success, marker="o", color="tab:green")
axes[2].axhline(0.8, color="k", ls="--", lw=1, label="критерий приёмки")
axes[2].set_ylim(-0.05, 1.05)
axes[2].set_title("Eval/Success Rate — доля замкнутых кругов")
axes[2].set_xlabel("шаг сбора"); axes[2].grid(alpha=0.3); axes[2].legend()

plt.tight_layout()
plt.show()

print(f"итоговая оценка: награда {result.last_eval_reward:.3f}, "
      f"кругов {result.last_eval_success:.0%}")
print(f"итоговая температура α: {float(algo.alpha):.4f}")

THRESHOLD = 0.80   # блок success_criteria конфига, метрика Eval/Success Rate

if QUICK_RUN:
    print(f"быстрый прогон: критерий приёмки ({THRESHOLD:.0%}) не проверяется — "
          "проверялась работоспособность конвейера")
else:
    assert result.last_eval_success >= THRESHOLD, (
        f"не достигнут критерий приёмки: {result.last_eval_success:.0%} < {THRESHOLD:.0%}")
    print(f"критерий приёмки достигнут: {result.last_eval_success:.0%} >= {THRESHOLD:.0%}")

итоговая оценка: награда 13.034, кругов 100%
итоговая температура α: 0.0380
быстрый прогон: критерий приёмки (80%) не проверяется — проверялась работоспособность конвейера


C:\Users\-\AppData\Local\Temp\ipykernel_22972\3755042117.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Экспорт в ONNX и верификация

Экспортируется актор; критики существовали только ради его обучения.

Тонкость именно этого метода: обёртка контракта делит выход на 3
($\mathrm{clamp}(x,-3,3)/3$), а политика SAC выдаёт $\tanh(\mu) \in (-1,1)$.
Поэтому `policy_module()` возвращает актора, **домноженного на 3**, — и в графе
оказывается ровно $\tanh(\mu)$. Домножение безопасно: $3\tanh(\mu)$ никогда
не достигает границ обрезки, и деление восстанавливает значение точно.

In [9]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из реального распределения среды (требование 10.5), а не шум.
collected = [[] for _ in vec.obs_shapes]
obs_now = vec.reset()
while sum(len(c) for c in collected[0]) < 64:
    for sensor, batch in enumerate(obs_now):
        collected[sensor].append(batch.copy())
    obs_now = vec.step(algo.deterministic_action(vec.flatten_obs(obs_now))).obs
sample = [np.concatenate(parts, axis=0)[:64] for parts in collected]

spec = ActionSpecLite(continuous_size=ACTION_DIM)
policy = algo.policy_module()
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
graph_action = session.run(["deterministic_continuous_actions"],
                           {"obs_0": sample[0], "obs_1": sample[1]})[0]
python_action = algo.deterministic_action(np.concatenate(sample, axis=1))
gap = float(np.abs(graph_action - python_action).max())
print(f"\nmax|ONNX - Python| по действию: {gap:.3e}")
assert gap < 1e-5, "оценка в Python и граф расходятся — инференс в Unity будет другим"

if not QUICK_RUN:
    # Модель проекта обновляется только полным прогоном (T-14).
    model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                  "Models" / f"{ENV_ID}_sac.onnx")
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    model_dest.write_bytes(onnx_path.read_bytes())
    print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
else:
    print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

vec.close()

OK    структура (onnx.checker)
OK    opset — ожидался 9, в модели 9
OK    имена входов — лишние=[], отсутствуют=[]
OK    имена выходов — лишние=[], отсутствуют=[]
OK    тип выхода continuous_actions — ожидался float32, получен elem_type=1 (float32)
OK    тип выхода deterministic_continuous_actions — ожидался float32, получен elem_type=1 (float32)
OK    прогон onnxruntime, батч 1
OK    прогон onnxruntime, батч 64
OK    константа version_number — ожидалось [3.0], получено [3.0]
OK    константа memory_size — ожидалось [0.0], получено [0.0]
OK    константа continuous_action_output_shape — ожидалось [2.0], получено [2.0]
OK    числовой паритет: deterministic_continuous_actions — max|ONNX-PyTorch| = 2.384e-07, допуск 1e-04
OK    диапазон непрерывных действий — [-0.9601, 0.8104], допустимо [-1, 1] (clamp ±3 / 3)
ИТОГ: ПРОЙДЕНО (13/13)

max|ONNX - Python| по действию: 1.788e-07
быстрый прогон: модель в проект Unity НЕ копируется (T-14)


## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E07_RacingCar__sac.yaml --python-reward <оценка из раздела 8>
```

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E07_RacingCar/Scenes/E07_RacingCar.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента `Car`: `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: машина проходит круг по контрольным точкам, не задевая стен.

### Если награда в Unity заметно ниже

Первое, что проверяют для SAC, — что экспортирован именно `tanh(μ)`,
а не сырое μ: `policy_module()` обязан возвращать `export_module()`.
Дальше — общий список причин в `docs/07_TROUBLESHOOTING.md`.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, **↑/↓** — газ и тормоз, **←/→** — руль.